<a href="https://colab.research.google.com/github/peakodev/data_science/blob/main/home_12/Hw12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 12

Зробіть summary нижчевказаного тексту використовуючи бібліотеки для NLP: nltk та SpaCy



In [1]:
text = 'The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. She had the honor of being chosen as the Return to Flight vehicle after both the Challenger and Columbia accidents. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. Discovery also carried the Hubble Space Telescope to orbit and performed two of the five servicing missions to the observatory. She flew the first and last dedicated Department of Defense (DoD) missions, as well as the first unclassified defense-related mission. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. She was the first orbiter to dock to the ISS, and the first to perform an exchange of a resident crew. Under Criterion C, Discovery is significant as a feat of engineering. According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.” Although her base structure followed a conventional aircraft design, she used advanced materials that both minimized her weight for cargo-carrying purposes and featured low thermal expansion ratios, which provided a stable base for her Thermal Protection System (TPS) materials. The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield. Other notable engineering achievements of the orbiter included the first reusable orbital propulsion system, and the first two-fault-tolerant Integrated Avionics System. As Hale stated, the Space Shuttle remains “the largest, fastest, winged hypersonic aircraft in history,” having regularly flown at twenty-five times the speed of sound.'

# Experiment

In [2]:
import string
import re

import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from heapq import nlargest

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


In [3]:
nlp = spacy.load('en_core_web_sm')

In [4]:
punctuation_to_remove = string.punctuation.replace(".", "")

preprocessed_text = re.sub(f"[{re.escape(punctuation_to_remove)}]", " ", text).lower()

sentence_tokens = sent_tokenize(preprocessed_text)
word_tokens = word_tokenize(preprocessed_text)

stop_words = set(stopwords.words('english'))
word_tokens = [word for word in word_tokens if word not in stop_words]

In [5]:
len(word_tokens), len(sentence_tokens)

(306, 16)

In [6]:
preprocessed_text

'the orbiter discovery  ov 103  is considered eligible for listing in the national register of historic places  nrhp  in the context of the u.s. space shuttle program  1969 2011  under criterion a in the areas of space exploration and transportation and under criterion c in the area of engineering. because it has achieved significance within the past fifty years  criteria consideration g applies. under criterion a  discovery is significant as the oldest of the three extant orbiter vehicles constructed for the space shuttle program  ssp   the longest running american space program to date  she was the third of five orbiters built by nasa. unlike the mercury  gemini  and apollo programs  the ssp’s emphasis was on cost effectiveness and reusability  and eventually the construction of a space station. including her maiden voyage  launched august 30  1984   discovery flew to space thirty nine times  more than any of the other four orbiters  she was also the first orbiter to fly twenty missi

In [7]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentence_tokens)
sentence_scores = tfidf_matrix.sum(axis=1)

In [8]:
ratio = 0.3
select_length = int(len(sentence_tokens) * ratio)
summary_indices = nlargest(select_length, range(len(sentence_scores)), key=sentence_scores.__getitem__)
summary = [sentence_tokens[i] for i in summary_indices]

In [9]:
summary = ' '.join(summary)

print(summary)

according to wayne hale  a flight director from johnson space center  the space shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable  winged  hypersonic  cargo carrying spacecraft.” although her base structure followed a conventional aircraft design  she used advanced materials that both minimized her weight for cargo carrying purposes and featured low thermal expansion ratios  which provided a stable base for her thermal protection system  tps  materials. including her maiden voyage  launched august 30  1984   discovery flew to space thirty nine times  more than any of the other four orbiters  she was also the first orbiter to fly twenty missions. under criterion a  discovery is significant as the oldest of the three extant orbiter vehicles constructed for the space shuttle program  ssp   the longest running american space program to date  she was the third of five orbiters built by nasa. the orbiter discovery  ov 103  is considered 

# Inference (result service) with cosine_similarity

In [37]:
import numpy as np
import spacy

import nltk
from nltk.tokenize import sent_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from heapq import nlargest


class Summarizer:

    ratio = 0.3

    def __init__(self):
        nltk.download('averaged_perceptron_tagger')
        self.nlp = spacy.load('en_core_web_sm')
        self.vectorizer = TfidfVectorizer()


    def __call__(self, text):
        # Preprocess the text
        sentences, lemmatized_words = self.__preprocess_text(text)

        # Calculate sentence scores
        score = self.__get_score(lemmatized_words)

        # Get diversity-aware summary
        summary = self.__make_summary(sentences, score)

        return summary


    def __preprocess_text(self, text):

        # Tokenize sentences using NLTK
        sentences = sent_tokenize(text)

        # Lemmatize and clean the text using SpaCy
        lemmatized_sentences = []
        for sentence in sentences:
            doc = nlp(sentence)
            lemmatized_sentence = ' '.join([token.lemma_ for token in doc if not token.is_stop and not token.is_punct])
            lemmatized_sentences.append(lemmatized_sentence)

        return sentences, lemmatized_sentences


    def __get_score(self, sentence_tokens):

        # Compute TF-IDF matrix
        tfidf_matrix = self.vectorizer.fit_transform(sentence_tokens)

        # Compute cosine similarity to detect sentence importance
        similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

        # Score sentences by summing their similarity scores (gives more importance to diverse sentences)
        sentence_scores = np.sum(similarity_matrix, axis=1)

        return sentence_scores


    def __make_summary(self, original_sentences, sentence_scores):

        # Calculate the number of sentences to select
        select_length = int(len(original_sentences) * self.ratio)

        # Use heapq.nlargest to get the indices of the top ratio sentences
        top_sentence_indices = nlargest(select_length, range(len(sentence_scores)), key=sentence_scores.__getitem__)

        # Sort by sentence order to keep it coherent
        top_sentence_indices.sort()

        # Build the final summary from the selected sentences
        summary = [original_sentences[i] for i in top_sentence_indices]

        return ' '.join(summary)


summarizer = Summarizer()

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [38]:
summary = summarizer(text)
print(summary)

The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle.


In [39]:
len(text), len(summary)

(2906, 934)